# Generating or Converting 6D FLIM Data as 5D Modulo FLIM Data

Generating a small synthetic 6D data (or converting from a 6D .zarr file) to be converted to 5D Modulo dataset (similar to the way described [here](https://ome-model.readthedocs.io/en/stable/developers/6d-7d-and-8d-storage.html#d-7d-and-8d-storage))



## Environment settings

Run the following commands to create a conda environment and install the necessary packages to run this notebook

```bash
mamba create -n ome-zarr-env python=3.10 ome-zarr

pip install bioio

pip install ome-types bioio-ome-tiff

## Import packages

In [19]:
import os
import zarr
import tifffile
import ome_types
import ome_zarr
import bioio_ome_tiff
import tifffile
import numpy as np

from pathlib import Path
from ome_types import to_xml, from_xml
from ome_types.model import OME, Image, Pixels, Channel, TiffData
from bioio import BioImage
from ome_zarr.io import parse_url
from ome_zarr.writer import write_image
from ome_zarr.reader import Reader

In [2]:
print("- ome_types version: ", ome_types.__version__)
print("- bioio version: ", bioio_ome_tiff.__version__)
print("- tifffile version: ", tifffile.__version__)
print("- zarr version: ", zarr.__version__)

- ome_types version:  0.5.2
- bioio version:  1.0.1
- tifffile version:  2024.9.20
- zarr version:  2.18.3


- ome-zarr:                  0.9.0

Other possible useful packages not installed

- bioio-ome-zarr:            1.0.1
- napari-ome-zarr:           0.6.1

## Generate or Load 6D FLIM Data

### Synthetic Random Data

In [3]:
name = 'Synthetic_6D_FLIM_data_reshaped_to_5D_modulo'
flim_size = 2
original_6D_shape = (2, 2, 2, 2, 16, 16)
new_chunk_shape = tuple([2, 2, 2, 8, 8])
voxel_resolution = (0.5, 0.27, 0.27)

In [4]:
# Create some dummy image data (6D: C, H, T, Z, Y, X)
data = np.random.randint(0, 256, original_6D_shape, dtype=np.uint8)
print(data.shape)

(2, 2, 2, 2, 16, 16)


### Load 6D `.zarr` FLIM data

In [13]:
# zarr array converted using napari-flim-phasor-plotter plugin
data_path = Path("/home/pol_haase/mazo260d/Data/I227_Lifetime_Unmixing_of_Dyes_with_Overlapping_Sprectra/TMR17_1_sptw/TMR17_1_sptw.zarr")
name = "TMR17_1"
voxel_resolution = (0.5, 0.27, 0.27)

In [14]:
data = zarr.open(data_path, mode='r')
original_6D_shape = data.shape
flim_size = data.shape[1]
new_chunk_shape = tuple([data.shape[1], 1, 8, 32, 32])
print(data.shape)

(2, 279, 45, 37, 256, 256)


Re-shape data to 5D with a specific axes order (this may load all data to memory, so consider where to run this)

In [15]:
# Put time axis first
data = np.moveaxis(data, [0, 1, 2, 3, 4, 5], [0, 2, 1, 3, 4, 5]) # CHTZYX -> CTHZYX (swap T and H to make T come first)
# Merge time and histogram (flim photon counts) axes
data = np.reshape(data, (data.shape[0], data.shape[1]*data.shape[2], data.shape[3], data.shape[4], data.shape[5])) # C[TH]ZYX Merge T and H dimensions
# Put merged_time axis first
data = np.moveaxis(data, [0, 1, 2, 3, 4], [1, 0, 2, 3, 4]) # C[TH]ZYX -> [TH]CZYX (needed by ome_zarr)
print(data.shape)


(12555, 2, 37, 256, 256)


Create ome metadata with ome-types

In [16]:
# Create OME metadata without the structured annotations
# ome-types expects xy to come first
def create_ome_metadata(data_shape):
    t_size, c_size, z_size, y_size, x_size = data_shape
    channels = [Channel(id=f"Channel:{i}", name=f"Channel {i}") for i in range(c_size)]
    pixels = Pixels(
        dimension_order="XYZTC",
        type="uint8",
        size_t=t_size,
        size_c=c_size,
        size_z=z_size,
        size_y=y_size,
        size_x=x_size,
        physical_size_x=voxel_resolution[-1],
        physical_size_y=voxel_resolution[-2],
        physical_size_z=voxel_resolution[-3],
        channels=channels,
        tiff_data=[TiffData()]
    )
    image = Image(id="Image:0", name=name, pixels=pixels)
    ome = OME(images=[image])
    return ome

# Generate OME metadata
ome_metadata = create_ome_metadata(data.shape)
ome_xml = to_xml(ome_metadata)
print(ome_xml)

<OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd">
  <Image ID="Image:0" Name="TMR17_1">
    <Pixels ID="Pixels:1" DimensionOrder="XYZTC" Type="uint8" SizeX="256" SizeY="256" SizeZ="37" SizeC="2" SizeT="12555" PhysicalSizeX="0.27" PhysicalSizeY="0.27" PhysicalSizeZ="0.5">
      <Channel ID="Channel:0" Name="Channel 0"/>
      <Channel ID="Channel:1" Name="Channel 1"/>
    </Pixels>
  </Image>
</OME>



/tmp/ipykernel_29375/202674164.py:25: UserWarning: Unrecognized fields for type <class 'ome_types._autogenerated.ome_2016_06.pixels.Pixels'>: {'tiff_data'}
  ome_metadata = create_ome_metadata(data.shape)


Add "Modulo" structured annotation

In [17]:

# Add the structured annotations XML to the OME-XML
structured_annotation = f"""
<StructuredAnnotations>
    <XMLAnnotation ID="Annotation:3" Namespace="openmicroscopy.org/omero/dimension/modulo">
        <Value>
            <Modulo namespace="http://www.openmicroscopy.org/Schemas/Additions/2011-09">
                <ModuloAlongT Type="lifetime" TypeDescription="TCSPC" Start="0" Step="1" End="{flim_size-1}"/>
            </Modulo>
        </Value>
    </XMLAnnotation>
</StructuredAnnotations>
"""

# Insert the structured annotation into the OME-XML
# This should be added just before the closing </OME> tag
ome_xml_with_sa = ome_xml.replace("</OME>", structured_annotation + "</OME>")
print(ome_xml_with_sa)

<OME xmlns="http://www.openmicroscopy.org/Schemas/OME/2016-06" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.openmicroscopy.org/Schemas/OME/2016-06 http://www.openmicroscopy.org/Schemas/OME/2016-06/ome.xsd">
  <Image ID="Image:0" Name="TMR17_1">
    <Pixels ID="Pixels:1" DimensionOrder="XYZTC" Type="uint8" SizeX="256" SizeY="256" SizeZ="37" SizeC="2" SizeT="12555" PhysicalSizeX="0.27" PhysicalSizeY="0.27" PhysicalSizeZ="0.5">
      <Channel ID="Channel:0" Name="Channel 0"/>
      <Channel ID="Channel:1" Name="Channel 1"/>
    </Pixels>
  </Image>

<StructuredAnnotations>
    <XMLAnnotation ID="Annotation:3" Namespace="openmicroscopy.org/omero/dimension/modulo">
        <Value>
            <Modulo namespace="http://www.openmicroscopy.org/Schemas/Additions/2011-09">
                <ModuloAlongT Type="lifetime" TypeDescription="TCSPC" Start="0" Step="1" End="278"/>
            </Modulo>
        </Value>
    </XMLAnnotation>
</StructuredAnnotations>


Optionally save as ome-tiff

In [ ]:
# Save the 5D data as an OME-TIFF file with the updated OME-XML metadata
# file_path = '5D_image_with_sa.ome.tif'
# tifffile.imwrite(
#     file_path,
#     data,
#     metadata={"axes": "XYZTC"},
#     description=ome_xml_with_sa,  # Attach the modified OME-XML with structured annotation
# )


Save locally as ome-zarr

In [18]:

file_path = name + ".zarr"
os.mkdir(file_path)

store = parse_url(file_path, mode="w").store
root = zarr.group(store=store)
# ome_zarr.write expects time to come first.
write_image(image=data, group=root, axes="tczyx", storage_options=dict(chunks=new_chunk_shape))

[]

## Read the recently created file back

In [20]:
# read the image data
store = parse_url(file_path, mode="r").store

reader = Reader(parse_url(file_path))
# nodes may include images, labels etc
nodes = list(reader())
# first node will be the image pixel data
image_node = nodes[0]

dask_data = image_node.data

dask_data

/home/pol_haase/mazo260d/miniforge3/envs/ome-zarr-env/lib/python3.10/site-packages/zarr/creation.py:614: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


[dask.array<from-zarr, shape=(12555, 2, 37, 256, 256), dtype=uint16, chunksize=(279, 1, 8, 32, 32), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(12555, 2, 37, 128, 128), dtype=uint16, chunksize=(279, 1, 8, 32, 32), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(12555, 2, 37, 64, 64), dtype=uint16, chunksize=(279, 1, 8, 32, 32), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(12555, 2, 37, 32, 32), dtype=uint16, chunksize=(279, 1, 8, 32, 32), chunktype=numpy.ndarray>,
 dask.array<from-zarr, shape=(12555, 2, 37, 16, 16), dtype=uint16, chunksize=(279, 1, 8, 16, 16), chunktype=numpy.ndarray>]

Check the first element of the list (highest resolution ?)

In [ ]:
dask_data[0]

In [ ]:
dask_data[1]